In [1]:
import torch
from transformers import AutoModel, AutoTokenizer
import torch.nn as nn
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

/opt/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
df_clara = pd.read_csv("../datasets/training-v1/offenseval-training-v1.tsv", sep='\t')
df_clara.columns = ["id", "text", "label_A", "label_B", "label_C"]

labelsA = df_clara["label_A"].map({"NOT": 0, "OFF": 1}).values

df_claraB = df_clara[df_clara["label_A"] == "OFF"]
labelsB = df_claraB["label_B"].map({"TIN": 1, "UNT": 0}).values

df_claraC = df_clara[(df_clara["label_A"] == "OFF") & (df_clara["label_B"] == "TIN")]
labelsC = df_claraC["label_C"].map({"IND": 0, "GRP": 1, "OTH": 2}).values

df_claraA = df_clara["text"].values.tolist()
df_claraB = df_claraB["text"].values.tolist()
df_claraC = df_claraC["text"].values.tolist()
df_claraA[0:10]

['@USER She should ask a few native Americans what their take on this is.',
 '@USER @USER Go home you’re drunk!!! @USER #MAGA #Trump2020 👊🇺🇸👊 URL',
 'Amazon is investigating Chinese employees who are selling internal data to third-party sellers looking for an edge in the competitive marketplace. URL #Amazon #MAGA #KAG #CHINA #TCOT',
 '@USER Someone should\'veTaken" this piece of shit to a volcano. 😂"',
 '@USER @USER Obama wanted liberals &amp; illegals to move into red states',
 '@USER Liberals are all Kookoo !!!',
 '@USER @USER Oh noes! Tough shit.',
 '@USER was literally just talking about this lol all mass shootings like that have been set ups. it’s propaganda used to divide us on major issues like gun control and terrorism',
 '@USER Buy more icecream!!!',
 '@USER Canada doesn’t need another CUCK! We already have enough #LooneyLeft #Liberals f**king up our great country! #Qproofs #TrudeauMustGo']

In [ ]:
sentences = [
    "I love all people, no matter where they come from.",
    "That group is disgusting and should not exist.",
    "We should build a more inclusive and respectful community.",
    "You don't belong here. Go back to your country."
]

In [ ]:
class Paola(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased", num_outputs=8, bin_outputs=5):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_outputs)
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, bin_outputs),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.regressor(pooled), self.classifier(pooled)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_paola = Paola().to(device)
model_paola.load_state_dict(torch.load("model2_loaded.pth", map_location=device))

print("model2_loaded.pth loaded and ready to use!")

2025-04-30 17:58:05.512426: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


model2_loaded.pth loaded and ready to use!


In [14]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
encodings = tokenizer(df_claraA[0:10], truncation=True, padding=True, max_length=128, return_tensors="pt")

model_paola.eval()
input_ids = encodings['input_ids'].to(device)
attention_mask = encodings['attention_mask'].to(device)

with torch.no_grad():
    preds_num, preds_bin = model_paola(input_ids=input_ids, attention_mask=attention_mask)

preds_num = preds_num.cpu().numpy()
preds_bin = preds_bin.cpu().numpy()
preds_bin = (preds_bin > 0.5).astype(int)

for idx, sentence in enumerate(df_claraA[0:5]):
    print(f"Sentence: {sentence}")
    print(f"Numerical predictions: {preds_num[idx]}")
    print(f"Binary predictions: {preds_bin[idx]}")
    print()

Sentence: @USER She should ask a few native Americans what their take on this is.
Numerical predictions: [2.2887914  1.9238985  1.4340876  1.2148466  2.0133154  0.9041642
 1.8230721  0.12435535]
Binary predictions: [1 0 0 0 0]

Sentence: @USER @USER Go home you’re drunk!!! @USER #MAGA #Trump2020 👊🇺🇸👊 URL
Numerical predictions: [3.6650374 3.5250227 3.2195625 2.927325  2.9489949 1.8697491 3.0327158
 0.3712163]
Binary predictions: [0 0 0 0 0]

Sentence: Amazon is investigating Chinese employees who are selling internal data to third-party sellers looking for an edge in the competitive marketplace. URL #Amazon #MAGA #KAG #CHINA #TCOT
Numerical predictions: [2.4486656  2.1668115  1.7653397  1.4392807  2.0595467  0.94233704
 2.1723251  0.11320253]
Binary predictions: [0 0 1 0 0]

Sentence: @USER Someone should'veTaken" this piece of shit to a volcano. 😂"
Numerical predictions: [3.4436996 3.4225473 3.1440768 2.836907  2.8938067 2.3831418 3.0396724
 0.9530526]
Binary predictions: [0 0 0 0 0]



In [ ]:
class ColineA(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased", extra_features_dim=12, outputs=2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        # Optional: freeze BERT if you want to use the pretrained weights only
        for param in self.bert.parameters():
            param.requires_grad = False

        # New classifier that takes [BERT CLS output + 12 features]
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + extra_features_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, outputs),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask, extra_features):
        bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = bert_outputs.last_hidden_state[:, 0]  # CLS token output

        # Concatenate with the extra features
        x = torch.cat((pooled_output, extra_features), dim=1)

        return self.classifier(x)


In [ ]:
# Load existing model
old_model = ClaraA()
old_model.load_state_dict(torch.load("model_claraA_loaded.pth", map_location=device))

# New model
new_model = ColineA().to(device)

# Copy BERT weights
new_model.bert.load_state_dict(old_model.bert.state_dict())


In [ ]:
# Example tensors (replace with your data)
# input_ids, attention_mask already defined
my_extra_features = np.concatenate([preds_num, preds_bin], axis=1)
extra_features = torch.tensor(my_extra_features).float().to(device)  # shape: [batch_size, 12]
labels = torch.tensor(labelsA).float().to(device)  # shape: [batch_size, 2] if multi-label

# Dataset and DataLoader
dataset = TensorDataset(input_ids, attention_mask, extra_features, labels)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Model
model = ColineA().to(device)

# Loss and optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

# Training
for epoch in range(3):
    model.train()
    for input_ids_batch, attn_mask_batch, extra_batch, label_batch in loader:
        optimizer.zero_grad()
        outputs = model(input_ids_batch, attn_mask_batch, extra_batch)
        loss = criterion(outputs, label_batch)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} done.")


In [ ]:
class ModelOlid(nn.Module):
    def __init__(self, task, model_name=None, num_labels=None, class_weights=None):
        super(ModelOlid, self).__init__()
        self.task = task
        self.model_name = model_name or MODEL_NAMES[task]
        self.num_labels = num_labels or NUM_LABELS[task]
        self.class_weights = class_weights

        self.bert = AutoModel.from_pretrained(self.model_name)
        hidden_size = self.bert.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + 12, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, self.num_labels)
        )

    def forward(self, input_ids, attention_mask, extra_feats):
        # BERT outputs: (last_hidden_state, pooler_output, hidden_states, attentions)
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]  # CLS token

        # Concatenate with extra features
        combined = torch.cat((pooled_output, extra_feats), dim=1)

        logits = self.classifier(combined)
        return logits
